# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuguda999/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question:** Which content pages should a client's SEO/content team review first for refresh, and why — given limited review capacity and real search demand data?

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring (picked in ML-02, confirmed through ML-10).

**The decision this supports:** an editor with ~250-1,000+ candidate pages per client (ML-10) cannot review them all. This work produces a ranked, reason-coded queue so the highest-value review happens first.

**Cost of a wrong call:** a false positive wastes reviewer time; a false negative lets a real, demand-backed decline go unnoticed. Both costs are why a *ranked, explained* queue beats either a flat list or a black-box score.

**Why data/ML at all:** the starter baseline rules already show real signal (ML-07's two CONFIRMED bucket tests), but they check one signal at a time. Whether a learned model earns its complexity on top of that rule was the open question this capstone actually tested — and the honest answer, found in ML-08/09, was no. That negative result is itself the finding worth reporting.

In [1]:
# Section 1 is narrative. The capacity/demand numbers it references are backed by code
# in w01_research_question.ipynb (ML-02) and re-verified with fresh numbers in Section 6 below.

## 2. Data

**Release:** `FlyRank/internship-warehouse` (Hugging Face, gated, pseudonymized). Table: `fact_content_daily_performance`, `month=2026-03` partition — a mid-panel month, chosen per the dataset guide's iteration rule (the sealed final month, June 2026, was never touched).

**Grain and window:** one row = one (client, content item, day). `report_date` spans 2026-03-01 → 2026-03-31, split internally: days 1-15 as the feature window, days 16-31 as the label window — a genuine past→future split inside one month (verified in ML-04).

**Excluded, and why (public-safe):**
- GA4 engagement columns — only ~4.2% of March rows have `ga4_data_available = TRUE`; a raw GA4 feature would mostly encode "does this client have GA4 wired up," not real engagement.
- Any raw client name, domain, URL, or query — never in the release to begin with; only pseudonym hash ids (`client_hash_id`, `content_hash_id`) are used, for grouping/joining only, never as features.
- FlyRank's own product flags (health score, quick-win tags) — not shipped in the dataset by design, and would be circular if they were: they encode a decision the product already made.

Verified below: grain holds (zero duplicate keys), row count and date span match the release, and the GA4-availability number above is a measured fact, not an assumption.

In [2]:
import os
import getpass
import duckdb
import pandas as pd
import numpy as np

# find the repo root from wherever this kernel started (VS Code/Colab/CLI all differ)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
con.execute("SET http_timeout=300")
con.execute("SET http_retries=5")
con.execute("SET http_retry_wait_ms=1000")
con.execute("SET http_retry_backoff=2")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

# grain + counts + availability, verified fresh (same checks as ML-04)
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) c
    FROM read_parquet('{MONTH}') GROUP BY 1, 2, 3 HAVING c > 1 LIMIT 5
""").df()
contract = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM read_parquet('{MONTH}')
""").df()

print(f"duplicate keys found: {len(grain_check)} (0 means the grain holds)")
print(f"row_count={contract['row_count'][0]:,}  window={contract['min_date'][0]} to {contract['max_date'][0]}")
print(f"ga4_data_available=TRUE: {contract['ga4_available_rows'][0]:,} rows "
      f"({contract['ga4_available_rows'][0] / contract['row_count'][0] * 100:.1f}%)")

duplicate keys found: 0 (0 means the grain holds)
row_count=9,841,378  window=2026-03-01 00:00:00 to 2026-03-31 00:00:00
ga4_data_available=TRUE: 413,966 rows (4.2%)


## 3. Methodology

**Label:** `is_declining = imp_last < 0.8 × imp_prev`, where `imp_prev` sums `gsc_impressions` over days 1-15 and `imp_last` sums it over days 16-31. A proxy, not a confirmed real-world outcome — but a genuine past→future split, not a same-window trick.

**Features (5, all closed over days 1-15 only):** `imp_prev`, `clk_prev`, `ctr_prev` (= clicks/impressions), `avg_position_prev`, `active_days_prev`. `imp_last` is never a feature — only the label input.

**Baseline (ML-07):** `score = imp_prev × max(0, expected_ctr_for_position_tier − ctr_prev)`, eligible only where `imp_prev ≥ 100` and `0 < avg_position_prev ≤ 20`. Two signals behind it were bucket-tested and CONFIRMED before trusting the rule: CTR-vs-position (33.7% vs 20.4% decline rate, n≈31k each side) and volume (40.2% → 28.3% decline rate across terciles, n=150,675).

**Model (ML-08):** Logistic Regression and Random Forest (Gradient Boosting as a stretch), same features, evaluated at precision@K because the decision is "review the top K first," not "classify everyone."

**Validation design:** `GroupShuffleSplit` on `client_hash_id` — grouped, not random-by-row. A random row split lets the model see other pages from the same client during training and partly memorize the client instead of the pattern.

**Leakage checks (ML-04, ML-09):** timeline drawn (feature window strictly before label window); population filter (`imp_prev > 0`) uses only the feature window; no product flags anywhere; the "confession test" — add the exact ratio the label is computed from, watch ROC-AUC jump to 1.000, remove it, back to the honest number — run and passed on the final feature set.

In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feat = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_prev,
        SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_clicks ELSE 0 END) AS clk_prev,
        AVG(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_prev,
        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15' AND gsc_impressions > 0 THEN report_date END) AS active_days_prev,
        SUM(CASE WHEN report_date > DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) AS imp_last
    FROM read_parquet('{MONTH}')
    GROUP BY 1, 2
    HAVING SUM(CASE WHEN report_date <= DATE '2026-03-15' THEN gsc_impressions ELSE 0 END) > 0
""").df()

feat["ctr_prev"] = feat["clk_prev"] / feat["imp_prev"]
feat["is_declining"] = (feat["imp_last"] < 0.8 * feat["imp_prev"]).astype(int)
feat = feat.dropna(subset=["avg_position_prev"]).reset_index(drop=True)
feat["log_imp_prev"] = np.log1p(feat["imp_prev"])
feat["log_clk_prev"] = np.log1p(feat["clk_prev"])

FEATURES = ["log_imp_prev", "log_clk_prev", "ctr_prev", "avg_position_prev", "active_days_prev"]
print(f"shape: {feat.shape[0]:,} rows | clients: {feat['client_hash_id'].nunique()} | base rate: {feat['is_declining'].mean():.3f}")

# leakage confession test, on the honest grouped split
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
tr_idx, te_idx = next(gss.split(feat[FEATURES], feat["is_declining"], feat["client_hash_id"]))

def fit_auc(cols):
    m = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1)
    m.fit(feat.loc[tr_idx, cols], feat.loc[tr_idx, "is_declining"])
    p = m.predict_proba(feat.loc[te_idx, cols])[:, 1]
    return roc_auc_score(feat.loc[te_idx, "is_declining"], p)

auc_without = fit_auc(FEATURES)
feat["future_ratio_LEAKY"] = feat["imp_last"] / feat["imp_prev"]
auc_with = fit_auc(FEATURES + ["future_ratio_LEAKY"])
feat = feat.drop(columns=["future_ratio_LEAKY"])
print(f"confession test: without suspect AUC={auc_without:.3f}, with suspect AUC={auc_with:.3f} -> removed, back to {auc_without:.3f}")

shape: 150,675 rows | clients: 44 | base rate: 0.326


confession test: without suspect AUC=0.578, with suspect AUC=1.000 -> removed, back to 0.578


## 4. Results (vs baseline)

The headline finding, and it's a negative result: **on this grouped test split, none of the three trained models beat the hand-written baseline rule at any K.** The baseline wins or ties every column; Random Forest's own test ROC-AUC sits around 0.58 — barely above a coin flip.

That is not a failure of the project. It's the "don't reward complexity alone" result the whole track was built to surface honestly. The table and chart below are regenerated fresh in this run and saved as `work/outputs/model_vs_baseline_metrics.json` — the receipt these numbers trace back to.

In [4]:
import json
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

X, y = feat[FEATURES], feat["is_declining"]
X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_s, y_train)
rf = RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_train, y_train)
gb = GradientBoostingClassifier(n_estimators=200, max_depth=3, random_state=42).fit(X_train, y_train)

test_df = feat.iloc[te_idx].copy()
test_df["p_logreg"] = logreg.predict_proba(X_test_s)[:, 1]
test_df["p_rf"] = rf.predict_proba(X_test)[:, 1]
test_df["p_gb"] = gb.predict_proba(X_test)[:, 1]

# baseline rule, thresholds fit on TRAIN only, applied to test (same recipe as ML-08)
train_eligible = (feat.iloc[tr_idx]["imp_prev"] >= 100) & (X_train["avg_position_prev"] > 0) & (X_train["avg_position_prev"] <= 20)
tier = pd.cut(X_train.loc[train_eligible, "avg_position_prev"], bins=[0, 3, 10, 20], labels=["1-3", "4-10", "11-20"]).astype(str)
expected_ctr_by_tier = X_train.loc[train_eligible].assign(position_tier=tier).groupby("position_tier")["ctr_prev"].median().to_dict()

def expected_ctr_for_position(pos):
    if pos <= 3:
        return expected_ctr_by_tier["1-3"]
    if pos <= 10:
        return expected_ctr_by_tier["4-10"]
    return expected_ctr_by_tier["11-20"]

test_df["expected_ctr"] = test_df["avg_position_prev"].apply(expected_ctr_for_position)
test_eligible = (test_df["imp_prev"] >= 100) & (test_df["avg_position_prev"] > 0) & (test_df["avg_position_prev"] <= 20)
gap = np.where(test_eligible, test_df["expected_ctr"] - test_df["ctr_prev"], 0.0)
test_df["score_baseline"] = np.where(test_eligible, test_df["imp_prev"] * np.clip(gap, 0, None), 0.0)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

Ks = [10, 20, 50, 100]
rows = []
for name, col in [("baseline (ML-07 rule)", "score_baseline"), ("logistic_regression", "p_logreg"),
                   ("random_forest", "p_rf"), ("gradient_boosting", "p_gb")]:
    row = {"method": name}
    for k in Ks:
        row[f"precision@{k}"] = round(precision_at_k(test_df[col], test_df["is_declining"], k), 3)
    rows.append(row)
comparison = pd.DataFrame(rows)
test_base_rate = float(test_df["is_declining"].mean())
print(f"test base rate: {test_base_rate:.3f}  (test rows: {len(test_df):,}, test clients: {test_df['client_hash_id'].nunique()})")
print(comparison.to_string(index=False))

with open("work/outputs/model_vs_baseline_metrics.json", "w") as f:
    json.dump({
        "month": "2026-03", "split": "GroupShuffleSplit(client_hash_id, test_size=0.25, random_state=42)",
        "test_rows": int(len(test_df)), "test_clients": int(test_df["client_hash_id"].nunique()),
        "test_base_rate": round(test_base_rate, 4),
        "rf_test_roc_auc": round(float(roc_auc_score(y_test, test_df["p_rf"])), 4),
        "comparison": comparison.to_dict(orient="records"),
    }, f, indent=2)
print("wrote work/outputs/model_vs_baseline_metrics.json")

fig, ax = plt.subplots(figsize=(8, 4.5))
x = np.arange(len(Ks))
width = 0.2
for i, row in comparison.iterrows():
    vals = [row[f"precision@{k}"] for k in Ks]
    ax.bar(x + (i - 1.5) * width, vals, width, label=row["method"])
ax.axhline(test_base_rate, color="black", linestyle="--", linewidth=1, label="base rate")
ax.set_xticks(x)
ax.set_xticklabels([f"precision@{k}" for k in Ks])
ax.set_ylabel("precision")
ax.set_title("Model vs baseline, honest grouped split (March 2026)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("work/figures/model_vs_baseline.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("wrote work/figures/model_vs_baseline.png")

test base rate: 0.366  (test rows: 14,426, test clients: 11)
               method  precision@10  precision@20  precision@50  precision@100
baseline (ML-07 rule)           0.5          0.60          0.56           0.54
  logistic_regression           0.4          0.45          0.38           0.42
        random_forest           0.3          0.35          0.44           0.45
    gradient_boosting           0.5          0.50          0.50           0.44
wrote work/outputs/model_vs_baseline_metrics.json


wrote work/figures/model_vs_baseline.png


## 5. Limitations

- **One mid-panel month.** Everything here is built and scored on March 2026 only. Not checked against a different month, season, or the sealed final month.
- **One proxy label.** `is_declining` is a within-month days-1-15-vs-16-31 impression ratio, not a confirmed multi-month outcome.
- **44 clients, one lane.** Nothing here has been checked on other clients, verticals, or lanes.
- **~50% of rows are `thin_evidence`** (fewer than 15/15 active days) — a real feature of a mixed-traffic portfolio, not a bug, but it means half the slice can't be scored with confidence by this method.
- **Decision-support only, never causal.** "Review this page" is not "refreshing it will recover traffic" — that claim needs an experiment (a before/after refresh test), which this cross-sectional data cannot provide.
- **The model did not beat the baseline here.** This is reported as the finding, not softened — see Section 4.
- **Retrospective, not a live forecast.** The model and rule are fit and scored on the same March slice as a demonstration of the mechanics; this is not a claim about future months.

In [5]:
thin_share = (feat["active_days_prev"] < 15).mean()
print(f"rows with fewer than 15/15 active days (thin_evidence): {thin_share * 100:.1f}%")

rows with fewer than 15/15 active days (thin_evidence): 54.7%


## 6. Ranked recommendations

Five archetypes (ML-10), each a reason code with one action — built entirely from the validated baseline and model above, no unvalidated clustering. Loaded below from the committed receipt (`work/outputs/action_playbook_metrics.json`).

| Archetype | Action | Who acts | Never automate |
|---|---|---|---|
| `low_ctr_for_position` | `review_ctr_fix` | Editor rewrites title/snippet | Never auto-publish the rewrite |
| `high_volume_poor_position` | `flag_for_deeper_review` | Editor diagnoses root cause | No auto-remedy exists for this one |
| `model_flags_declining_no_baseline_signal` | `monitor_closely` | Editor watchlists | Model score isn't reliable enough to act alone |
| `thin_evidence` | `wait_for_more_data` | No one, yet | Never score-and-act on thin evidence |
| `monitor_default` | `monitor` | No one | — |

In [6]:
with open("work/outputs/action_playbook_metrics.json") as f:
    playbook_metrics = json.load(f)

archetype_table = pd.DataFrame(playbook_metrics["archetypes"]).T.sort_values("tier")
archetype_table.index.name = "archetype"
print(f"month={playbook_metrics['month']}  n_rows={playbook_metrics['n_rows']:,}  "
      f"actionable_now={playbook_metrics['actionable_now_pct']}%")
archetype_table

month=2026-03  n_rows=150,675  actionable_now=22.8%


,n,decline_rate,action,tier
archetype,,,,
low_ctr_for_position,30974,0.337,review_ctr_fix,1
high_volume_poor_position,3372,0.5294,flag_for_deeper_review,2
model_flags_declining_no_baseline_signal,7682,0.3935,monitor_closely,3
thin_evidence,76142,0.3468,wait_for_more_data,4
monitor_default,32505,0.2308,monitor,5


## 7. Artifacts the paper embeds

Three figures and two metrics receipts, all committed to `work/figures/` and `work/outputs/`. The deployed paper's Results and Recommendations sections embed these directly — confirmed present below.

In [7]:
artifacts = [
    "work/figures/model_vs_baseline.png",
    "work/figures/archetype_decline_rates.png",
    "work/outputs/model_vs_baseline_metrics.json",
    "work/outputs/action_playbook_metrics.json",
]
for path in artifacts:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"{'OK ' if exists else 'MISSING'}  {path}  ({size:,} bytes)")

OK   work/figures/model_vs_baseline.png  (50,260 bytes)
OK   work/figures/archetype_decline_rates.png  (53,406 bytes)
OK   work/outputs/model_vs_baseline_metrics.json  (871 bytes)
OK   work/outputs/action_playbook_metrics.json  (1,007 bytes)


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [x] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom. Live at https://yuguda999.github.io/flyrank-ml-internship-starter/ — verified 200, correct title, flyrank.ai link, both chart images load, matches local HEAD (f77f80b).
- [ ] **ML-12 done in this notebook's closing cells:** not part of this task — separate card, not requested yet.